# Backtest — 지난 N일 히스토리로 신호 엔진 검증

주의: 과거 성과는 미래 수익을 보장하지 않습니다. 이 결과는 지표가 **데이터에 존재했던 규칙을 얼마나 포착하는지** 확인하는 검증용이지, 수익 시뮬레이션이 아닙니다.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

from crypto_analysis.backtest import run as run_backtest
from crypto_analysis.storage import read_parquet

perp = read_parquet('deribit_ohlcv').sort_values('ts').reset_index(drop=True)
spot = read_parquet('binance_spot_btc_1h')
funding = read_parquet('deribit_funding')
len(perp), len(spot), len(funding)

In [ ]:
result = run_backtest(
    perp_ohlcv=perp,
    spot_ohlcv=spot,
    funding_df=funding,
    ticker_hist=None,
    hold_hours=8,
    step_hours=4,
)

print(f'trades: {len(result.trades)}')
print(f'hit rate: {result.hit_rate:.2%}')
print(f'avg fwd return: {result.avg_fwd_return:+.3%}')
print(f'sharpe (annualised): {result.sharpe:.2f}')
result.trades.tail()

In [ ]:
if not result.equity_curve.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=result.equity_curve.index, y=result.equity_curve.values, mode='lines', name='equity'))
    fig.update_layout(title='Backtested equity curve', yaxis_title='multiple')
    fig.show()

In [ ]:
if not result.trades.empty:
    print(result.trades['verdict'].value_counts())
    print(result.trades.groupby('verdict')['pnl'].describe())